# Exploración y verificación de la API de Series de Tiempo (datos.gob.ar)

## Fuentes y referencias
- **API de Series de Tiempo (endpoint base):** `https://apis.datos.gob.ar/series/api/series`
- **Documentación de la API:** https://datosgobar.github.io/series-tiempo-ar-api/
- **Referencia del recurso `/series` (parámetros):** https://datosgobar.github.io/series-tiempo-ar-api/reference/api_reference/
- **Catálogo de metadatos (CSV):** https://apis.datos.gob.ar/series/api/dump/series-tiempo-metadatos.csv
- **Repositorio oficial de la API:** https://github.com/datosgobar/series-tiempo-ar-api
- **Precios de la API de Claude (Haiku 5.5):** https://platform.claude.com/docs/en/about-claude/pricing


Los parámetros de la API utilizados en este proyecto son `ids` (con sufijos de transformación, por ejemplo `id:sum:percent_change`), `representation_mode`, `collapse`, `collapse_aggregation`, `start_date`, `end_date`, `limit`, `sort` y `format`.

## Contexto y objetivo
Este notebook es la primera etapa de un proyecto de fine-tuning con QLoRA (Unsloth) de un modelo Qwen de tamaño reducido. El sistema objetivo es un agente que responde preguntas en lenguaje natural consultando la API de Series de Tiempo de la República Argentina. El proyecto compara el desempeño del agente con el modelo base y con el modelo fine-tuneado, con observabilidad en Langfuse. El entrenamiento y los scripts se ejecutan en Google Colab con almacenamiento en Google Drive.

Dado que la calidad de un dataset de entrenamiento depende de la fuente que lo respalda, esta etapa documenta el comportamiento de la API y la calidad de sus datos. El resultado es un catálogo verificado contra la API y un conjunto de decisiones de diseño para el dataset, las herramientas del agente y la evaluación.

Datos consultados en noviembre de 2026. La API y el catálogo se actualizan con el tiempo, por lo que los números de este documento corresponden a esa fecha.

## Resumen de resultados
- El catálogo contiene 34.599 series. Se definió un recorte de 20.631 (seis temas económicos) y se verificaron contra la API las 8.209 con fecha final declarada en 2025 o 2026.
- 6.710 series (81,7%) devolvieron datos y 1.499 (18,3%) devolvieron `count: 0`, aunque el catálogo indica que poseen valores.
- Los campos de fecha y de valor del catálogo no son una fuente confiable: la fecha final difiere de la de la API en 30,7% de las series con datos, y en un grupo de series la API entrega solo un tramo inicial de la serie.
- Quedaron 6.521 series utilizables para entrenamiento y 1.685 reservadas como casos de "sin datos" para evaluación (186 truncadas y 1.499 vacías).

## Procedimiento
1. Prueba de la API (`/series`): responde 200 con datos, metadatos y URL de descarga.
2. Descarga del catálogo de metadatos (34.599 series, 29 columnas). La lectura directa con `pd.read_csv(url)` devolvió 403; con `requests` y un `User-Agent` de navegador funcionó. Hipótesis no verificada: el servidor filtra por identificador del cliente.
3. Corrección de codificación: los acentos se leían incorrectamente y se resolvió leyendo los bytes con `encoding="utf-8"`.
4. Perfilado del catálogo y definición del recorte: Precios, Sector Externo, Dinero y Bancos, Actividad, Empleo e Ingresos y Finanzas Públicas. Se excluyeron las series discontinuadas, las series `apis_*` (uso de las propias APIs) y el tema Delitos, por tratarse de un dominio sensible y de naturaleza distinta.
5. Verificación contra la API de las 8.209 series con final en 2025 o 2026 (4 hilos, reintentos, guardado incremental por lotes de 400; duración aproximada: 47 minutos). Todas respondieron con status 200.
6. Comparación de fechas y valores entre catálogo y API, y clasificación de cada serie por estado.
7. Controles de integridad y de comportamiento de la API (errores, duplicados, títulos repetidos, series truncadas).
8. Almacenamiento de snapshots en Drive.

## Hallazgos

### Identificadores y búsqueda
- Los ids son crípticos (por ejemplo, `168.1_T_CAMBIOR_D_0_0_26`) y existen decenas de miles. No son memorizables por un modelo pequeño, por lo que el agente debe buscarlos. El fine-tuning debe orientarse al flujo (buscar, elegir, construir la consulta) y no a los ids.
- Títulos repetidos: 4.347 de las 6.521 series de entrenamiento (66,7%) comparten título con otra serie. Dentro de un mismo dataset permanecen repetidas 4.146 (63,6%). La combinación `(dataset_id, distribucion_id, serie_titulo)` identifica a todas (0 repetidas). El campo `distribucion_titulo` aporta la frecuencia y la base de precios (corrientes, constantes o índice).
- Los metadatos pueden contradecirse: en un caso observado, la distribución indica "precios constantes" y `serie_unidades` indica "millones de pesos corrientes". Se observó un único ejemplo y no se midió su frecuencia.
- Existen 133 datasets; los 10 más grandes concentran 2.234 series (34,3%).

### Series sin datos
- 1.499 de 8.209 series (18,3%) responden 200 con `count: 0`. Un re-chequeo de 100 de ellas, realizado el mismo día, confirmó que seguían vacías; no hay información sobre su evolución temporal.
- La proporción varía por tema, desde 11,7% (Empleo e Ingresos) hasta 22,4% (Dinero y Bancos). En series diarias es de 45,6%, sobre un grupo reducido.

### Fechas (6.707 series con datos y fecha válida)
- La fecha final coincide en 3.920 series; la API contiene datos más recientes en 2.601 (típicamente 60 a 90 días) y más antiguos en 186 (174 por más de un año). Dado que el promedio del desfase (-99,5 días) está distorsionado por valores extremos, se reportan mediana y percentiles.
- En series anuales el desfase es infrecuente (0,4%); en mensuales y trimestrales ocurre en aproximadamente 43%.
- Rezago del catálogo (muestra de 8 series): en 5 el CSV fuente llega entre 2 y 3 meses más allá de la fecha final del catálogo; en 3 coincide.

### Series truncadas (muestra de 9 series)
- En las 186 series donde la API termina antes que el catálogo, la API entrega un tramo inicial continuo y omite el resto. En las 8 series de la muestra con CSV fuente comparable, el CSV llega considerablemente más lejos que la API. En 7 de 8, la cantidad de valores devueltos por la API coincide exactamente con los meses entre el inicio del CSV y el último dato de la API; la octava es una serie diaria y no se comprobó de ese modo.
- Caso extremo: `174.1_AGADOS_M3_0_0_28`, cuya API termina en 1958 mientras el catálogo declara 2026.
- Causa desconocida. El problema reside en la API y no en el catálogo.
- Limitación: el truncamiento solo se detecta por comparación con la fecha del catálogo. Una serie clasificada como `normal` podría estar truncada sin ser detectada.

### Comportamiento de la API
- Los usos incorrectos (id inexistente, `collapse` inválido o más fino que la serie, `representation_mode` inválido, fecha mal formada) devuelven status 400 con un mensaje descriptivo en JSON. Ejemplo para `collapse` más fino que la serie: "Intervalo de collapse inválido... Pruebe con un intervalo mayor".
- Una fecha fuera de rango devuelve 200 con `data: []`, sin indicación de error.
- Inconsistencia: tres series diarias (`331.2_*`) devuelven un `count` de aproximadamente 5.000 pero `data: []` al consultar con `limit=1&sort=desc`, mientras que con `sort=asc` devuelven datos. Por este motivo no se pudo determinar su fecha final (estado `sin_fecha`).

### Valores del catálogo frente a la API (series con desfase 0, n = 3.920)
- Coinciden en 94,7% con tolerancia 1e-6 y en 98,7% dentro de 1%. Se detectaron errores del catálogo: tres series distintas de un mismo dataset (`41.1_*`) comparten el mismo `serie_valor_ultimo`. Los campos numéricos del catálogo no se utilizan para etiquetas ni para búsqueda.
- La columna `distribucion_url_descarga` está vacía en las 6.521 series de entrenamiento. La URL está disponible en la respuesta de la API (`meta.distribution.downloadURL`).

### Estados de las series con datos

| Estado | Series | Uso en el proyecto |
|---|---|---|
| `normal` | 4.647 | Entrenamiento y evaluación |
| `api_adelantada` | 1.874 | Entrenamiento y evaluación, con las fechas de la API |
| `api_truncada` | 186 | Evaluación de casos "sin datos para el período" |
| `sin_fecha` | 3 | Casos especiales (fallan con `sort=desc`) |

Series de entrenamiento por frecuencia: mensuales 2.617, anuales 1.996, trimestrales 1.816, semestrales 62 y diarias 30. Las dos últimas categorías se tratan como casos especiales por su bajo número.

## Decisiones de diseño
- **Fuente de verdad:** la API. Las etiquetas se obtienen ejecutando la consulta. El catálogo se emplea solo para el texto de búsqueda (título, `distribucion_titulo`, descripción, dataset, tema, unidades y frecuencia), sin campos numéricos.
- **Herramientas del agente:** `search_series`, `get_series` y `submit_answer`. Las respuestas de error de la API se devuelven sin modificar para permitir la corrección por parte del modelo.
- **Partición:** por `dataset_id` (entrenamiento, validación y prueba), con un máximo de preguntas por dataset para evitar el predominio de los datasets más grandes.
- **Datos congelados:** caché de pedidos con clave en los parámetros normalizados. Se consulta primero la caché; ante una ausencia, se consulta la API real, se almacena la respuesta y se registra el fallo. Cada evaluación reporta su tasa de fallos de caché.
- **Preguntas ambiguas:** se incluyen únicamente en la evaluación. No se entrena la conducta de solicitar aclaración; se mide el comportamiento de cada modelo ante este tipo de preguntas.
- **Respuesta ante series truncadas o vacías:** "la API no devuelve datos para ese período", dado que la fuente original sí puede contenerlos.
- **Tipos de pregunta:** valor puntual, último dato, variación, agregación, comparación, recuperación de errores, sin datos y ambigua (esta última solo en evaluación). En las preguntas de agregación, la función (`sum`, `avg`, `end_of_period`) se explicita en el enunciado.
- **Línea base:** el modelo base se evalúa con un prompt básico y con un prompt optimizado.
- **Generación de preguntas:** las etiquetas se calculan por código; un modelo de lenguaje se utiliza únicamente para parafrasear el enunciado, con filtros automáticos sobre período, indicador y tipo de agregación.

## Verificación de acceso al modelo generador de paráfrasis
- Se verificó la conectividad con Claude Haiku 5.5 (`claude-haiku-5-5`) con la clave almacenada en los Secrets de Colab.
- Una prueba de paráfrasis (118 tokens de entrada y 95 de salida) tuvo un costo de USD 0,000059, a las tarifas de USD 0,10 y USD 0,50 por millón de tokens para prompts de hasta 100.000 tokens.
- La prueba mostró un riesgo de contenido: una de las tres paráfrasis reemplazó "tipo de cambio" por "dólar", expresión ambigua en el contexto argentino, y no fue detectada por un filtro que solo verificaba año y mes. Los filtros deben validar además el indicador y el tipo de agregación.

## Notas metodológicas y correcciones
- Las respuestas vacías se interpretaron inicialmente como un caso límite de la API; el catálogo indicaba que esas series contenían datos, por lo que se trataron como una discrepancia a investigar.
- Una muestra de 15 series sugería que el catálogo era confiable en fechas; sobre las 6.707 series, 30,7% difiere en más de 31 días.
- La muestra de 200 series estimó 16% de series vacías; el total arrojó 18,3%.
- El tema Actividad concentraba 15 de las 32 fallas de la muestra, pero su tasa (15,2%) es inferior al promedio; el resultado reflejaba su mayor cantidad de series.
- Una comparación inicial de valores arrojó 80,3% de coincidencia por incluir series con fechas finales distintas; restringida a desfase 0, la coincidencia es de 94,7%.
- Una tabla de casos extremos estaba ordenada por una columna incorrecta y se corrigió.
- Las tres series `sin_fecha` fueron inicialmente descartadas bajo el criterio de ser diarias; el motivo correcto es la inconsistencia de la API con `sort=desc`.

## Limitaciones y puntos abiertos
- No se verificaron las series con final anterior a 2025 ni las 40 con final en 2027 o 2126 (posibles errores de carga, no investigados).
- Cada serie se consultó una única vez; se desconoce si las series vacías o truncadas cambian con el tiempo.
- Se desconoce la causa de las series vacías y truncadas.
- El corte de "serie vigente" (2025-01-01) es una aproximación y no un criterio validado.
- La licencia de los datos debe confirmarse antes de redistribuirlos en un repositorio público.
- Que el modelo fine-tuneado supere al modelo base con prompt optimizado y buscador no está demostrado; es la hipótesis que el proyecto evaluará.

## Archivos generados (Drive: `qlora-series-ar/data/`)
- `catalogo_completo.csv`: catálogo completo (34.599 series).
- `catalogo_recorte.csv`: recorte de 20.631 series.
- `verificacion_series.csv`: resultado de la API para las 8.209 series verificadas.
- `catalogo_verificado.csv`: 6.710 series con datos, con columnas `desfase_dias` y `estado`.
- `series_sin_datos.csv`: 1.499 series sin datos en la API.

## Reproducción
Ejecutar las celdas en orden. Si el runtime se reinicia, ejecutar la celda de reconstrucción (monta Drive y carga `catalogo_recorte.csv`) y luego la de verificación, que retoma desde lo almacenado.

## Próximos pasos
1. Buscador del catálogo (SQLite FTS5) y partición por `dataset_id`.
2. Medición del recall del buscador con preguntas derivadas de las plantillas.
3. Generador de preguntas, paráfrasis con filtros automáticos y caché de pedidos.
4. Línea base del agente con el modelo base, con trazas en Langfuse.
5. Fine-tuning QLoRA con Unsloth y evaluación comparativa.

---
**Ficha**

| | |
|---|---|
| **Runtime** | CPU (sin GPU) |
| **Entradas** | Ninguna: descarga el catálogo de metadatos y consulta la API de Series de Tiempo |
| **Salidas** (`MyDrive/qlora-series-ar/data/`) | `catalogo_completo.csv`, `catalogo_recorte.csv`, `catalogo_verificado.csv`, `series_sin_datos.csv`, `verificacion_series.csv` |
| **Costo y tiempo** | Sin costo (API pública). Las verificaciones hacen miles de consultas a la API, así que la duración depende de la red |
| **Orden** | Primero. Lo siguen `00b` y `01` |
| **En el repositorio** | Sin módulo equivalente todavía: el catálogo resultante es la entrada de `scripts/particionar.py` |

**Advertencias.** Los resultados corresponden a la API y al catálogo consultados en noviembre de 2026; ambos pueden cambiar.


In [ ]:
import requests
import pandas as pd

BASE = "https://apis.datos.gob.ar/series/api/series"

# 1) Una consulta simple: tipo de cambio, últimos 5 datos
r = requests.get(BASE, params={
    "ids": "168.1_T_CAMBIOR_D_0_0_26",
    "limit": 5,
    "sort": "desc",
}, timeout=30)
print("Status:", r.status_code)
print(r.json())

# 2) Catálogo de series (metadatos)
URL_META = "https://apis.datos.gob.ar/series/api/dump/series-tiempo-metadatos.csv"
meta = pd.read_csv(URL_META)
print(meta.shape)
print(meta.columns.tolist())
meta.head()

In [ ]:
# Perfil rápido del catálogo
print("Frecuencias:")
print(meta["indice_tiempo_frecuencia"].value_counts(), "\n")

print("Temas (top 15):")
print(meta["dataset_tema"].value_counts().head(15), "\n")

print("Descontinuadas:")
print(meta["serie_discontinuada"].value_counts(dropna=False), "\n")

print("Actualizadas:")
print(meta["serie_actualizada"].value_counts(dropna=False), "\n")

# La serie que consultamos: ¿está actualizada?
fila = meta[meta["serie_id"] == "168.1_T_CAMBIOR_D_0_0_26"]
print(fila[["serie_titulo", "serie_indice_final", "serie_actualizada",
            "serie_discontinuada", "consultas_30_dias"]].T)

# Las 10 series más consultadas en los últimos 30 días
top = meta.sort_values("consultas_30_dias", ascending=False).head(10)
top[["serie_id", "serie_titulo", "indice_tiempo_frecuencia", "consultas_30_dias"]]

In [ ]:
import io, time, random
import requests
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

HEADERS = {"User-Agent": "Mozilla/5.0"}
URL_META = "https://apis.datos.gob.ar/series/api/dump/series-tiempo-metadatos.csv"

resp = requests.get(URL_META, headers=HEADERS, timeout=60)
meta = pd.read_csv(io.BytesIO(resp.content), encoding="utf-8", low_memory=False)

# Temas elegidos (verificá que los acentos ahora se vean bien)
TEMAS = ["Precios", "Sector Externo", "Dinero y Bancos", "Actividad",
         "Empleo e Ingresos", "Finanzas Públicas"]

cat = meta[
    meta["dataset_tema"].isin(TEMAS)
    & (meta["serie_discontinuada"] == False)
    & (~meta["serie_id"].str.startswith("apis_"))
].copy()
print("Series en el recorte:", len(cat))
print(cat["dataset_tema"].value_counts())

# Medir desfase catálogo vs API en una muestra de 15 series
muestra = cat.sample(15, random_state=42)
filas = []
for _, row in muestra.iterrows():
    rr = requests.get("https://apis.datos.gob.ar/series/api/series",
                      params={"ids": row["serie_id"], "limit": 1, "sort": "desc"},
                      headers=HEADERS, timeout=30)
    ultimo_api = rr.json()["data"][0][0] if rr.status_code == 200 and rr.json().get("data") else None
    filas.append({"serie_id": row["serie_id"],
                  "final_catalogo": row["serie_indice_final"],
                  "final_api": ultimo_api,
                  "status": rr.status_code})
    time.sleep(0.5)  # pausa para no saturar la API
pd.DataFrame(filas)

In [ ]:
import os

# 1) Las dos series que devolvieron vacío: ¿qué dice el catálogo?
vacias = ["362.1_INDICE_DIFNTOII2__60", "163.2_MPEL_CATON_0_0_55"]
print(meta[meta["serie_id"].isin(vacias)][
    ["serie_id", "serie_titulo", "serie_valores_cant",
     "serie_indice_inicio", "serie_indice_final", "serie_actualizada"]
].T)

# 2) Vigentes vs históricas
cat["final"] = pd.to_datetime(cat["serie_indice_final"], errors="coerce")
FECHA_CORTE = "2025-01-01"
cat["vigente"] = cat["final"] >= FECHA_CORTE

print("\nVigentes vs históricas:")
print(cat["vigente"].value_counts())
print("\nVigentes por frecuencia:")
print(cat[cat["vigente"]]["indice_tiempo_frecuencia"].value_counts())
print("\nVigentes por tema:")
print(cat[cat["vigente"]]["dataset_tema"].value_counts())

# 3) Guardar snapshot en Drive
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
os.makedirs(RUTA, exist_ok=True)

meta.to_csv(f"{RUTA}/catalogo_completo.csv", index=False, encoding="utf-8")
cat.to_csv(f"{RUTA}/catalogo_recorte.csv", index=False, encoding="utf-8")
print("\nGuardado en:", RUTA)
print(os.listdir(RUTA))

In [ ]:
# 1) Distribución del año final de las series del recorte
print("Año final de las series (top):")
print(cat["final"].dt.year.value_counts().sort_index(ascending=False).head(12))

# 2) Investigar las 2 series: probar distintas formas de consultar
BASE = "https://apis.datos.gob.ar/series/api/series"
for sid in ["163.2_MPEL_CATON_0_0_55", "362.1_INDICE_DIFNTOII2__60"]:
    print("\n=====", sid)
    pruebas = {
        "sin parámetros extra": {"ids": sid},
        "limit=3 desc":         {"ids": sid, "limit": 3, "sort": "desc"},
        "limit=3 asc":          {"ids": sid, "limit": 3, "sort": "asc"},
        "desde 2024-01":        {"ids": sid, "start_date": "2024-01"},
    }
    for nombre, params in pruebas.items():
        rr = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
        js = rr.json()
        datos = js.get("data", [])
        print(f"{nombre:22s} status={rr.status_code} filas={len(datos)} "
              f"count={js.get('count')} primeras={datos[:2]}")

In [ ]:
# 1) Inspeccionar las series con años raros
raras = cat[cat["final"].dt.year.isin([2027, 2126])]
print(raras[["serie_id", "serie_titulo", "serie_indice_final"]].head(10))

# 2) Respuesta cruda de una serie vacía (¿trae algún mensaje?)
rr = requests.get(BASE, params={"ids": "163.2_MPEL_CATON_0_0_55"},
                  headers=HEADERS, timeout=30)
print("\nRespuesta cruda:", rr.text[:600])

# 3) Muestra de 200 series con final en 2026: ¿cuántas devuelven datos?
pool = cat[cat["final"].dt.year == 2026]
muestra = pool.sample(200, random_state=1)

resultados = []
for _, row in muestra.iterrows():
    try:
        r2 = requests.get(BASE, params={"ids": row["serie_id"], "limit": 1, "sort": "desc"},
                          headers=HEADERS, timeout=30)
        n = len(r2.json().get("data", [])) if r2.status_code == 200 else -1
        st = r2.status_code
    except Exception as e:
        n, st = -1, str(e)[:40]
    resultados.append({"serie_id": row["serie_id"], "status": st, "filas": n,
                       "tema": row["dataset_tema"],
                       "frecuencia": row["indice_tiempo_frecuencia"]})
    time.sleep(0.3)

res = pd.DataFrame(resultados)
res["con_datos"] = res["filas"] > 0
print("\nStatus:\n", res["status"].value_counts())
print("\nCon datos:\n", res["con_datos"].value_counts())
print("\nSin datos por tema:\n", res[~res["con_datos"]]["tema"].value_counts())

In [ ]:
for nombre in ["cat", "meta", "BASE", "HEADERS", "RUTA"]:
    print(nombre, "OK" if nombre in globals() else "FALTA")

In [ ]:
import io, os, time
import requests
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}

# Recorte del catálogo (ya filtrado: 6 temas, sin discontinuadas, sin apis_*)
cat = pd.read_csv(f"{RUTA}/catalogo_recorte.csv", low_memory=False)
cat["final"] = pd.to_datetime(cat["serie_indice_final"], errors="coerce")

print("Series en el recorte:", len(cat))
print("Años finales (top):")
print(cat["final"].dt.year.value_counts().sort_index(ascending=False).head(5))
print("\nArchivos en la carpeta:", os.listdir(RUTA))

In [ ]:
from concurrent.futures import ThreadPoolExecutor

OUT = f"{RUTA}/verificacion_series.csv"

# Series a verificar: final en 2025 o 2026
pool = cat[cat["final"].dt.year.isin([2025, 2026])]
ids = pool["serie_id"].tolist()

# Retomar: saltear las ya verificadas
hechos = set(pd.read_csv(OUT)["serie_id"]) if os.path.exists(OUT) else set()
pendientes = [i for i in ids if i not in hechos]
print(f"Total: {len(ids)} | ya hechas: {len(hechos)} | pendientes: {len(pendientes)}")

def verificar(sid):
    status = None
    for intento in range(3):
        try:
            r = requests.get(BASE, params={"ids": sid, "limit": 1, "sort": "desc"},
                             headers=HEADERS, timeout=30)
            status = r.status_code
            if status == 200:
                js = r.json()
                data = js.get("data", [])
                return {"serie_id": sid, "status": 200,
                        "n_total_api": js.get("count"),
                        "ultimo_api": data[0][0] if data else None,
                        "ultimo_valor": data[0][1] if data else None}
        except Exception:
            status = -1
        time.sleep(1 + intento)   # espera creciente entre reintentos
    return {"serie_id": sid, "status": status, "n_total_api": None,
            "ultimo_api": None, "ultimo_valor": None}

# Procesar en lotes de 400, guardando después de cada lote
for i in range(0, len(pendientes), 400):
    lote = pendientes[i:i + 400]
    with ThreadPoolExecutor(max_workers=4) as ex:
        filas = list(ex.map(verificar, lote))
    pd.DataFrame(filas).to_csv(OUT, mode="a", index=False,
                               header=not os.path.exists(OUT))
    print(f"Lote listo: {min(i + 400, len(pendientes))}/{len(pendientes)}")

# Resumen
res = pd.read_csv(OUT).merge(
    cat[["serie_id", "dataset_tema", "indice_tiempo_frecuencia"]], on="serie_id")
res["con_datos"] = res["n_total_api"].fillna(0) > 0

print("\nStatus:\n", res["status"].value_counts())
print("\nCon datos:\n", res["con_datos"].value_counts())
print("\n% sin datos por tema:\n",
      (1 - res.groupby("dataset_tema")["con_datos"].mean()).round(3))
print("\n% sin datos por frecuencia:\n",
      (1 - res.groupby("indice_tiempo_frecuencia")["con_datos"].mean()).round(3))

In [ ]:
ver = pd.read_csv(f"{RUTA}/verificacion_series.csv")
ver["con_datos"] = ver["n_total_api"].fillna(0) > 0

# 1) Re-chequeo: ¿las series "sin datos" siguen sin datos?
sin = ver[~ver["con_datos"]]["serie_id"].sample(100, random_state=7).tolist()
siguen_vacias = 0
for sid in sin:
    rr = requests.get(BASE, params={"ids": sid, "limit": 1, "sort": "desc"},
                      headers=HEADERS, timeout=30)
    if rr.status_code == 200 and rr.json().get("count", 0) == 0:
        siguen_vacias += 1
    time.sleep(0.3)
print(f"Siguen vacías en el re-chequeo: {siguen_vacias}/100")

# 2) Catálogo verificado = recorte + resultado de la API
verif = cat.merge(
    ver[["serie_id", "n_total_api", "ultimo_api", "ultimo_valor", "con_datos"]],
    on="serie_id", how="inner")
ok = verif[verif["con_datos"]].copy()

# 3) ¿Coincide la fecha final del catálogo con la de la API?
ok["ultimo_api"] = pd.to_datetime(ok["ultimo_api"], errors="coerce")
ok["desfase_dias"] = (ok["ultimo_api"] - ok["final"]).dt.days
print("\nDesfase catálogo vs API (días):")
print(ok["desfase_dias"].describe())
print("Series con desfase mayor a 31 días:", (ok["desfase_dias"].abs() > 31).sum())

# 4) Qué quedó
print("\nVerificadas con datos:", len(ok))
print("\nPor tema:\n", ok["dataset_tema"].value_counts())
print("\nPor frecuencia:\n", ok["indice_tiempo_frecuencia"].value_counts())
print("\nFrecuencia de las series SIN datos:\n",
      verif[~verif["con_datos"]]["indice_tiempo_frecuencia"].value_counts())

# 5) Guardar en Drive
ok.to_csv(f"{RUTA}/catalogo_verificado.csv", index=False)
verif[~verif["con_datos"]].to_csv(f"{RUTA}/series_sin_datos.csv", index=False)
print("\nGuardado: catalogo_verificado.csv y series_sin_datos.csv")

In [ ]:
d = ok["desfase_dias"]

print("API más nueva que el catálogo (>0):", (d > 0).sum())
print("Coinciden (=0):", (d == 0).sum())
print("API más vieja que el catálogo (<0):", (d < 0).sum())
print("Extremos (menos de -365 días):", (d < -365).sum())

print("\nPercentiles:")
print(d.quantile([.01, .05, .10, .25, .50, .75, .90, .95, .99]))

print("\nMediana y % con |desfase| > 31 días, por frecuencia:")
print(ok.groupby("indice_tiempo_frecuencia")["desfase_dias"]
        .agg(mediana="median", pct_mayor_31=lambda s: (s.abs() > 31).mean()).round(3))

print("\nLos 10 casos más extremos:")
print(ok[d < -365][["serie_id", "serie_titulo", "final", "ultimo_api",
                    "n_total_api", "serie_valores_cant"]]
        .sort_values("final").head(10).to_string())

In [ ]:
# 1) Los casos más extremos de verdad (ordenados por desfase)
print(ok.sort_values("desfase_dias").head(10)[
    ["serie_id", "serie_titulo", "final", "ultimo_api",
     "n_total_api", "serie_valores_cant"]].to_string())

# 2) Estado de cada serie
def estado(x):
    if pd.isna(x): return "sin_fecha"
    if x < -31:    return "api_truncada"     # la API termina antes que el catálogo
    if x > 31:     return "api_adelantada"   # la API tiene datos más nuevos
    return "normal"

ok["estado"] = ok["desfase_dias"].apply(estado)
print("\n", ok["estado"].value_counts())
print("\n", pd.crosstab(ok["indice_tiempo_frecuencia"], ok["estado"]))

# 3) Guardar el catálogo con la nueva columna
ok.to_csv(f"{RUTA}/catalogo_verificado.csv", index=False)
print("\nGuardado con columna 'estado'")

In [ ]:
import numpy as np

# --- 6) Integridad ---
print("Duplicados en ver:", ver["serie_id"].duplicated().sum())
print("Duplicados en cat:", cat["serie_id"].duplicated().sum())

# --- 7) Las series sin fecha ---
print("\nSeries sin_fecha:")
print(ok[ok["estado"] == "sin_fecha"][["serie_id", "serie_titulo", "final",
                                       "ultimo_api", "n_total_api"]].to_string())

# --- 4 y 5) Metadatos, títulos duplicados y familias ---
ent = ok[ok["estado"].isin(["normal", "api_adelantada"])].copy()
print("\nSeries de entrenamiento:", len(ent))
print("Títulos repetidos (filas):", ent["serie_titulo"].duplicated(keep=False).sum())
print("Sin unidades:", ent["serie_unidades"].isna().sum())
print("Sin descripción:", ent["serie_descripcion"].isna().sum())
print("Datasets distintos:", ent["dataset_id"].nunique())
print("Series por dataset (top 10):")
print(ent["dataset_id"].value_counts().head(10))

# --- 3) Valor del catálogo vs. valor de la API (solo series 'normal') ---
n = ent[ent["estado"] == "normal"].copy()
n["v_cat"] = pd.to_numeric(n["serie_valor_ultimo"], errors="coerce")
n["v_api"] = pd.to_numeric(n["ultimo_valor"], errors="coerce")
comp = n.dropna(subset=["v_cat", "v_api"])
iguales = np.isclose(comp["v_cat"], comp["v_api"], rtol=1e-6)
print(f"\nValor último: comparables={len(comp)}, coinciden={iguales.sum()} "
      f"({iguales.mean():.1%}), último valor nulo en API={n['v_api'].isna().sum()}")

# --- 1) Respuestas de la API ante usos incorrectos ---
mensual = ent[(ent["estado"] == "normal") & (ent["indice_tiempo_frecuencia"] == "R/P1M")]["serie_id"].iloc[0]
anual = ent[(ent["estado"] == "normal") & (ent["indice_tiempo_frecuencia"] == "R/P1Y")]["serie_id"].iloc[0]
pruebas = {
    "id inexistente":                  {"ids": "no_existe_123"},
    "sin ids":                         {},
    "collapse inválido":               {"ids": mensual, "collapse": "decada"},
    "collapse más fino que la serie":  {"ids": anual, "collapse": "month"},
    "representation_mode inválido":    {"ids": mensual, "representation_mode": "xyz"},
    "fecha mal formada":               {"ids": mensual, "start_date": "2024-13-45"},
    "fecha fuera de rango":            {"ids": mensual, "start_date": "2090-01"},
}
print("\nRespuestas ante usos incorrectos:")
for nombre, params in pruebas.items():
    rr = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
    print(f"- {nombre}: status={rr.status_code} | {rr.text[:200]!r}")
    time.sleep(0.3)

# --- A) Las 3 series sin_fecha: ¿qué devuelve la API realmente? ---
for sid in ["331.2_COMPRAS_NENAL__39", "331.2_SALDO_BASERIA__15", "331.2_TOTAL_VARIION__27"]:
    for nombre, params in {"limit=1 desc": {"ids": sid, "limit": 1, "sort": "desc"},
                           "limit=3 asc":  {"ids": sid, "limit": 3, "sort": "asc"}}.items():
        rr = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
        js = rr.json()
        print(f"{sid} | {nombre} | status={rr.status_code} count={js.get('count')} "
              f"data={js.get('data', [])[:2]}")
        time.sleep(0.3)

# --- B) Valor del catálogo vs API, solo series con desfase exactamente 0 ---
c = ent[ent["desfase_dias"] == 0].copy()
c["v_cat"] = pd.to_numeric(c["serie_valor_ultimo"], errors="coerce")
c["v_api"] = pd.to_numeric(c["ultimo_valor"], errors="coerce")
c = c.dropna(subset=["v_cat", "v_api"])
c["dif_rel"] = (c["v_api"] - c["v_cat"]).abs() / c["v_cat"].abs().replace(0, np.nan)
print(f"\nComparables con desfase 0: {len(c)}")
for tol in [1e-6, 1e-3, 1e-2]:
    print(f"  coinciden con tolerancia {tol}: {(c['dif_rel'].fillna(0) <= tol).mean():.1%}")
print(c[c["dif_rel"] > 1e-3].sort_values("dif_rel", ascending=False).head(8)[
    ["serie_id", "serie_titulo", "v_cat", "v_api", "dif_rel"]].to_string())

# --- C) Títulos: ¿qué tan ambiguos son? ---
print("\nFilas con (dataset, título) repetido:",
      ent.duplicated(["dataset_id", "serie_titulo"], keep=False).sum())
print("\nTítulos más repetidos:")
print(ent["serie_titulo"].value_counts().head(10))

# --- D) URL de descarga: ¿cuántas están vacías en el catálogo? ---
print("\nURL de descarga vacía en el catálogo:", ent["distribucion_url_descarga"].isna().sum())

# --- E) Hipótesis de las truncadas, sacando la URL desde la API ---
sid = "78.3_IIP_0_A_26"
js = requests.get(BASE, params={"ids": sid, "limit": 1}, headers=HEADERS, timeout=30).json()
url = None
for m in js.get("meta", []):
    if isinstance(m, dict) and "distribution" in m:
        url = m["distribution"].get("downloadURL")
print("\nURL desde la API:", url)
if url:
    rr = requests.get(url, headers=HEADERS, timeout=60)
    print("CSV fuente: status", rr.status_code, "| bytes", len(rr.content))
    if rr.status_code == 200:
        d = pd.read_csv(io.BytesIO(rr.content), encoding="utf-8", low_memory=False)
        titulo = ok.loc[ok["serie_id"] == sid, "serie_titulo"].iloc[0]
        if titulo in d.columns:
            s = d[[d.columns[0], titulo]]
            print("Últimas filas:\n", s.tail(3).to_string())
            print("Último dato no nulo:\n", s.dropna().tail(2).to_string())
            print("La API llega hasta 1991-12-01 con 24 valores")
        else:
            print("No encontré la columna. Primeras columnas:", d.columns.tolist()[:8])

In [ ]:
pd.set_option("display.max_colwidth", 80)

# --- 1) Títulos repetidos: ¿qué campo los distingue? ---
rep = ent[ent.duplicated(["dataset_id", "serie_titulo"], keep=False)]
g = rep.groupby(["dataset_id", "serie_titulo"]).size().sort_values(ascending=False)
print("Grupos (dataset, título) repetidos:", len(g))
print(g.head(5))

d0, t0 = g.index[0]
ej = ent[(ent["dataset_id"] == d0) & (ent["serie_titulo"] == t0)]
print("\nEjemplo del grupo más grande:")
print(ej[["serie_id", "distribucion_titulo", "serie_descripcion", "serie_unidades"]].head(6).to_string())

claves = {
    "título": ["serie_titulo"],
    "dataset + título": ["dataset_id", "serie_titulo"],
    "dataset + distribución + título": ["dataset_id", "distribucion_id", "serie_titulo"],
    "... + descripción": ["dataset_id", "distribucion_id", "serie_titulo", "serie_descripcion"],
}
print("\nSeries sin clave única según los campos usados:")
for nombre, cols in claves.items():
    print(f"  {nombre}: {ent.duplicated(cols, keep=False).sum()}")

# --- 2) Truncadas: ¿el CSV fuente tiene más datos que la API? (8 series) ---
trunc = ok[ok["estado"] == "api_truncada"].sample(8, random_state=3)
cache, filas = {}, []
for _, row in trunc.iterrows():
    js = requests.get(BASE, params={"ids": row["serie_id"], "limit": 1},
                      headers=HEADERS, timeout=30).json()
    url = next((m["distribution"].get("downloadURL") for m in js.get("meta", [])
                if isinstance(m, dict) and "distribution" in m), None)
    primero = ultimo = n_csv = None
    if url:
        if url not in cache:
            r = requests.get(url, headers=HEADERS, timeout=60)
            cache[url] = pd.read_csv(io.BytesIO(r.content), low_memory=False) if r.status_code == 200 else None
            time.sleep(0.5)
        d = cache[url]
        if d is not None and row["serie_titulo"] in d.columns:
            s = d[[d.columns[0], row["serie_titulo"]]].dropna()
            primero, ultimo, n_csv = s.iloc[0, 0], s.iloc[-1, 0], len(s)
    filas.append({"serie_id": row["serie_id"],
                  "final_catalogo": str(row["final"].date()),
                  "ultimo_api": str(row["ultimo_api"].date()),
                  "n_api": row["n_total_api"],
                  "n_catalogo": row["serie_valores_cant"],
                  "primero_csv": primero, "ultimo_csv": ultimo, "n_csv": n_csv})
pd.DataFrame(filas)